<a href="https://colab.research.google.com/github/pejmanrasti/traitement_des_signaux_M2-/blob/main/05_MVU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MVU Estimators & Cramér–Rao Lower Bound (CRLB)
### Python Tutorial + Exercises

This notebook introduces the concepts of:
- MVU (Minimum Variance Unbiased) estimators  
- Fisher Information  
- Cramér–Rao Lower Bound (CRLB)  

We work through:
1. IID Gaussian model  
2. MVU estimator for DC level  
3. Fisher Information (numerically)  
4. Exponential model MVU estimator  

At the end, you will find **3 exercises** to practice.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")
np.random.seed(0)

In [ ]:
# True parameters
A_true = 2.0
sigma = 1.0
N = 1000

# Generate IID Gaussian noise
w = np.random.normal(0, sigma, N)

# Observations
x = A_true + w

plt.figure(figsize=(10,4))
plt.plot(x, label='x[n]')
plt.axhline(A_true, color='red', linestyle='--', label='True A')
plt.title("Generated Data: $x[n] = A + w[n]$")
plt.legend()
plt.show()

In [ ]:
A_hat = np.mean(x)

print("Estimated A (MVU) =", A_hat)
print("True A =", A_true)

In [ ]:
num_trials = 5000
A_estimates = []

for _ in range(num_trials):
    w = np.random.normal(0, sigma, N)
    x = A_true + w
    A_estimates.append(np.mean(x))

A_estimates = np.array(A_estimates)
empirical_var = np.var(A_estimates)
crlb = sigma**2 / N

print("Empirical variance =", empirical_var)
print("CRLB =", crlb)

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(A_estimates, bins=40, density=True, alpha=0.6, label="Estimator Distribution")
plt.axvline(A_true, color='red', linestyle='--', label="True A")
plt.title("Distribution of MVU Estimator vs CRLB")
plt.legend()
plt.show()

In [ ]:
def score_function(A, x, sigma):
    return np.sum(x - A) / (sigma**2)

num_trials = 2000
scores_squared = []

for _ in range(num_trials):
    w = np.random.normal(0, sigma, N)
    x = A_true + w
    s = score_function(A_true, x, sigma)
    scores_squared.append(s**2)

I_empirical = np.mean(scores_squared)
I_theoretical = N / sigma**2

print("Empirical Fisher Information =", I_empirical)
print("Theoretical Fisher Information =", I_theoretical)

# MVU Estimator for the Exponential Model

We consider the parametric signal model:

$$
x[n] = A\, r^n + w[n], \qquad w[n] \sim \mathcal{N}(0, \sigma^2),
$$

where \(A\) is the parameter to estimate and \(r\) is a known constant.

---

## **MVU Estimator for \(A\)**

For this model, the Minimum Variance Unbiased (MVU) estimator of \(A\) is:

$$
\hat{A} =
\frac{\displaystyle \sum_{n=0}^{N-1} r^n\, x[n]}
     {\displaystyle \sum_{n=0}^{N-1} r^{2n}}.
$$

This estimator is **unbiased** and **achieves the CRLB**, therefore it is **efficient**.

---

## **Let's simulate it below.**

In [ ]:
r = 0.9
N = 200

n = np.arange(N)
w = np.random.normal(0, sigma, N)
x = A_true * r**n + w

A_hat_exp = np.sum(r**n * x) / np.sum(r**(2*n))

print("Estimated A (exponential model) =", A_hat_exp)
print("True A =", A_true)

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(r**n * x, label="r^n * x[n]")
plt.title("Weighted Observations Used in MVU Estimator")
plt.legend()
plt.show()

# Exercises

---

## **Exercise 1 — Verify CRLB for the Exponential Model**

Consider the model:

$$
x[n] = A r^n + w[n], \qquad w[n] \sim \mathcal{N}(0, \sigma^2).
$$

The Cramér–Rao Lower Bound (CRLB) for estimating \(A\) is:

$$
\mathrm{Var}(\hat{A}) \ge \frac{\sigma^2}{\sum_{n=0}^{N-1} r^{2n}}.
$$

### **Tasks**
1. Simulate the model above for multiple independent trials.  
2. Compute the MVU estimator:

   $$
   \hat{A} = \frac{\sum_{n=0}^{N-1} r^n x[n]}{\sum_{n=0}^{N-1} r^{2n}}.
   $$

3. Estimate the empirical variance of \(\hat{A}\).  
4. Compare the empirical variance to the CRLB:

   $$
   \text{CRLB}(A) = \frac{\sigma^2}{\sum_{n=0}^{N-1} r^{2n}}.
   $$

5. Repeat for different values of

   $$
   r \in \{\,0.5,\; 0.9,\; 1.0,\; 1.1\,\}.
   $$

6. Plot empirical variance vs. theoretical CRLB as a function of \(r\).

---

## **Exercise 2 — Fisher Information for the Bernoulli Distribution**

Let the samples be IID Bernoulli:

$$
x[n] \sim \text{Bernoulli}(p), \qquad n = 0, \dots, N-1.
$$

The probability mass function is:

$$
P(x;n;p) = p^x (1 - p)^{1-x}, \qquad x \in \{0,1\}.
$$

The Fisher Information for parameter \(p\) is known to be:

$$
I(p) = \frac{N}{p(1-p)}.
$$

### **Tasks**
1. Write the log-likelihood:

   $$
   \ell(p) = \sum_{n=0}^{N-1} \Big[ x[n]\log p + (1 - x[n]) \log(1-p) \Big].
   $$

2. Compute the score function numerically using a finite difference approximation:

   $$
   \frac{\partial \ell(p)}{\partial p}
   \approx \frac{\ell(p+h) - \ell(p-h)}{2h}.
   $$

3. Estimate the Fisher Information empirically using:

   $$
   I_{\text{empirical}}(p) =
   \mathbb{E}\left[ \left( \frac{\partial \ell(p)}{\partial p} \right)^2 \right].
   $$

4. Compare the empirical estimate with the theoretical value:

   $$
   I(p) = \frac{N}{p(1-p)}.
   $$

5. Repeat the experiment for different values of \(p\), e.g.,  
   \(p \in \{0.1, 0.3, 0.5, 0.7, 0.9\}\).

---

## **Exercise 3 — Efficient Estimator with Correlated Noise**

Given:

$$
\mathbf{x} = A\mathbf{1} + \mathbf{w}, \quad
\mathbf{w} \sim \mathcal{N}(0, C),
$$

the efficient estimator is:

$$
\hat{A} = \frac{\mathbf{1}^T C^{-1} \mathbf{x}}{\mathbf{1}^T C^{-1}\mathbf{1}}.
$$

### Tasks:
1. Build the covariance matrix $begin:math:text$C$end:math:text$ where  
   $$
   C[i,j] = \rho^{|i-j|}
   $$
   with $begin:math:text$\\rho \= 0\.7$end:math:text$.  
2. Generate correlated Gaussian noise using Cholesky factorization.  
3. Simulate many estimates of $begin:math:text$\\hat\{A\}$end:math:text$.  
4. Compute empirical variance.  
5. Compare to the CRLB  
   $$
   \frac{1}{\mathbf{1}^T C^{-1}\mathbf{1}}.
   $$